# ML-10 — Content Action Playbook

**Lane:** Refresh / Content Opportunity Scoring  
**Decision supported:** Which eligible pages should an editor inspect first for a possible refresh?

This notebook turns the existing Week-5 grouped-validation result into a practical, human-reviewed queue. It rebuilds the same fixed client-held-out split from the repository starter CSV so the notebook can regenerate its queue and metrics. The target remains the trailing decline proxy (trend_direction equals down), not a future outcome. The model score is a ranking signal, not a calibrated probability or an instruction to edit.

The queue covers only the 30,000-row starter extract and the eligible subset with at least 100 impressions. Its rows are pseudonymous follow-up references; they are never printed here or committed as a CSV.


## 1. Ranked actions + reason codes

The first review tranche is the **top 50** items in the fixed client-held-out model ranking because Week 5 measured Precision@50 on this split. The remaining eligible held-out rows retain their rank as a lower-priority queue.

- TOP_50_MODEL_REVIEW: begin with a human check; it means top 50 by model ranking only.
- MODEL_RANKED_REVIEW: lower in the same held-out ranking; review only if capacity permits.
- Every row's action is editor_review. A reason code never means “refresh automatically.”

The model uses the same nine pre-decision content metadata fields as Week 5. The target and overlapping traffic-window fields are excluded from the feature set.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
REPO_CANDIDATES = [Path.cwd(), Path.cwd() / "flyrank-ml-internship", Path("/content/flyrank-ml-internship")]
REPO_ROOT = next((p for p in REPO_CANDIDATES if (p / DATA_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Clone Fiona's flyrank-ml-internship repository and run this notebook from its root.")

DATA_PATH = REPO_ROOT / DATA_REL
df = pd.read_csv(DATA_PATH)
assert df.shape == (30_000, 44), f"Unexpected starter extract shape: {df.shape}"
required = {
    "content_id", "client_id", "trend_direction", "impressions_90d",
    "search_volume", "competition", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "content_age_days", "days_since_last_update"
}
missing = sorted(required - set(df.columns))
assert not missing, f"Required fields missing: {missing}"

df["decline_proxy"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)
eligible = df.loc[pd.to_numeric(df["impressions_90d"], errors="coerce") >= 100].copy().reset_index(drop=True)
assert len(eligible) == 22_006

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update"
]
CATEGORICAL_FEATURES = ["content_type", "main_intent"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN = {
    "content_id", "client_id", "trend_direction", "trend_pct", "decline_proxy",
    "is_declining_label", "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
}
assert not (set(FEATURES) & FORBIDDEN), "A target, ID, or period-overlapping field entered the model."

X = eligible[FEATURES].copy()
y = eligible["decline_proxy"].to_numpy(dtype=int)
groups = eligible["client_id"].to_numpy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx])), "Client leakage between train and test."
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler())
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, NUMERIC_FEATURES),
    ("categorical", categorical_pipe, CATEGORICAL_FEATURES)
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
raw_model_scores = model.predict_proba(X_test)[:, 1]

def precision_at_k(labels, k):
    return float(np.mean(labels[:min(k, len(labels))]))

model_order = np.argsort(-raw_model_scores, kind="mergesort")
ranked_labels = y_test[model_order]
ranking_scores = raw_model_scores[model_order]
test_rows = eligible.iloc[test_idx].iloc[model_order].copy().reset_index(drop=True)

# Keep the frozen rule scores aligned to the original grouped test-row order.
rule_test_rows = eligible.iloc[test_idx]
rule_scores = (
    np.log1p(pd.to_numeric(rule_test_rows["impressions_90d"], errors="coerce").clip(lower=0).to_numpy())
    * pd.to_numeric(rule_test_rows["days_since_last_update"], errors="coerce").clip(lower=0, upper=180).to_numpy()
    / 180.0
)
rule_order = np.argsort(-rule_scores, kind="mergesort")
rule_labels = y_test[rule_order]

test_base_rate = float(y_test.mean())
model_metrics = {
    "precision_at_10": precision_at_k(ranked_labels, 10),
    "precision_at_50": precision_at_k(ranked_labels, 50),
    "average_precision": float(average_precision_score(y_test, raw_model_scores)),
    "roc_auc": float(roc_auc_score(y_test, raw_model_scores))
}
rule_metrics = {
    "precision_at_10": precision_at_k(rule_labels, 10),
    "precision_at_50": precision_at_k(rule_labels, 50),
    "average_precision": float(average_precision_score(y_test, rule_scores)),
    "roc_auc": float(roc_auc_score(y_test, rule_scores))
}
expected = {
    "test_rows": 4610, "test_clients": 8, "test_base_rate": 0.577,
    "model_precision_at_10": 0.600, "model_precision_at_50": 0.720,
    "model_average_precision": 0.632, "model_roc_auc": 0.571,
    "rule_precision_at_10": 0.500, "rule_precision_at_50": 0.380,
    "rule_average_precision": 0.521, "rule_roc_auc": 0.418
}
assert len(test_idx) == expected["test_rows"]
assert len(set(groups[test_idx])) == expected["test_clients"]
assert np.isclose(test_base_rate, expected["test_base_rate"], atol=0.001)
for key, observed in [
    ("model_precision_at_10", model_metrics["precision_at_10"]),
    ("model_precision_at_50", model_metrics["precision_at_50"]),
    ("model_average_precision", model_metrics["average_precision"]),
    ("model_roc_auc", model_metrics["roc_auc"]),
    ("rule_precision_at_10", rule_metrics["precision_at_10"]),
    ("rule_precision_at_50", rule_metrics["precision_at_50"]),
    ("rule_average_precision", rule_metrics["average_precision"]),
    ("rule_roc_auc", rule_metrics["roc_auc"])
]:
    assert np.isclose(observed, expected[key], atol=0.001), f"{key} changed: {observed:.3f}"

print(f"Starter extract: {len(df):,} rows × {df.shape[1]-1} source columns; eligible rows: {len(eligible):,}.")
print(f"Grouped split: {len(train_idx):,} train / {len(test_idx):,} test rows; {len(set(groups[train_idx]))}/{len(set(groups[test_idx]))} clients; no client overlap.")
print(f"Test proxy base rate: {test_base_rate:.3f}.")
print("Ranking metric                 Model     Week-4 rule")
for metric, label in [
    ("precision_at_10", "Precision@10"),
    ("precision_at_50", "Precision@50"),
    ("average_precision", "Average precision"),
    ("roc_auc", "ROC AUC")
]:
    print(f"{label:<30} {model_metrics[metric]:.3f}     {rule_metrics[metric]:.3f}")
print("Reproduction check: grouped model and same-row rule metrics match the Week-5 notebook.")


Starter extract: 30,000 rows × 44 source columns; eligible rows: 22,006.
Grouped split: 17,396 train / 4,610 test rows; 22/8 clients; no client overlap.
Test proxy base rate: 0.577.
Ranking metric                 Model     Week-4 rule
Precision@10                   0.600     0.500
Precision@50                   0.720     0.380
Average precision              0.632     0.521
ROC AUC                        0.571     0.418
Reproduction check: grouped model and same-row rule metrics match the Week-5 notebook.


In [2]:
days = pd.to_numeric(test_rows["days_since_last_update"], errors="coerce")
freshness = pd.cut(
    days, bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "366+"], right=True
).astype("string").fillna("unknown")

ranked_queue = test_rows[["client_id", "content_id", *FEATURES]].copy()
ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))
ranked_queue["ranking_score"] = ranking_scores
ranked_queue["freshness_bucket"] = freshness.to_numpy()
ranked_queue["reason_code"] = np.where(
    ranked_queue["rank"] <= 50, "TOP_50_MODEL_REVIEW", "MODEL_RANKED_REVIEW"
)
ranked_queue["action_label"] = "editor_review"
ranked_queue["review_note"] = np.where(
    ranked_queue["rank"] <= 50,
    "Measured top-50 tranche; inspect evidence before any edit.",
    "Lower-ranked in the same held-out queue; review only if capacity permits."
)

assert len(ranked_queue) == 4610
assert ranked_queue["content_id"].is_unique
assert ranked_queue["rank"].is_monotonic_increasing
assert ranked_queue["action_label"].eq("editor_review").all()
assert ranked_queue.loc[ranked_queue["rank"] <= 50, "reason_code"].eq("TOP_50_MODEL_REVIEW").all()

print(f"Export queue prepared: {len(ranked_queue):,} held-out rows; first tranche: 50.")
print("Queue rows carry pseudonymous follow-up IDs; IDs are not model inputs and are not printed or committed.")


Export queue prepared: 4,610 held-out rows; first tranche: 50.
Queue rows carry pseudonymous follow-up IDs; IDs are not model inputs and are not printed or committed.


## 2. Intended use and limits

**Intended user:** an editor or analyst deciding which eligible pages to inspect first. The queue prioritizes review using the Week-5 Logistic Regression ranking on the fixed client-held-out slice. A model score is not calibrated, and a high score is not proof that a page is declining.

The measured top-50 result was 0.720 against a 0.577 proxy base rate on one held-out split. That is a directional, sample-specific result for a trailing-window proxy; it does not establish future performance or show that a refresh will improve traffic. The Week-4 rule uses impressions from windows overlapping the proxy label, so it is retained only as a historical comparator.

**Cost/value:** review cost and realized refresh value were not measured. The practical value is a shorter, ordered review list; possible benefit is better use of editorial attention. Log review time, keep/refresh decisions, and outcomes on a later independent period before estimating cost per useful action or business value.


## 3. Human review + the no-go list

For each candidate, a human should verify the current page, search intent, factual accuracy, citations, technical status, and whether the page is already working well. Record the decision and the evidence used. An age bucket is a prompt to inspect freshness, not an instruction to rewrite.

**Never automate:** publishing or editing page copy; deleting or canonicalizing a page; making legal, medical, financial, or brand claims; inferring user intent or query-specific advice from absent query data; or treating a trailing proxy as a future outcome. Do not expose pseudonymous row IDs in a public report.


## 4. Monitoring / retrain triggers

These are proposed operating checks, not observed production thresholds:

1. Do not score if required columns, score inputs, or a usable reporting window are missing or malformed.
2. Re-run a client-grouped evaluation when a new independent outcome period is available; keep clients disjoint and compare with the current held-out result.
3. Pause use and investigate if Precision@50 falls below the 0.577 test base rate in a new independent window, or drops materially from the current 0.720 reference in two consecutive windows. The current reference is not a guarantee.
4. Retrain only after the target is defined from a future period and the new grouped/time-based split is documented. Keep the old model and metrics as the comparison.
5. Monitor missingness and score distribution by content type so input drift is visible; do not silently replace missing metadata with zero.


## 5. Exports for the paper

The notebook writes the ranked CSV and archetype-action CSV under work/outputs/; the repository deliberately ignores these generated CSVs. The small metrics JSON and SVG figure are committed as reproducible receipts for the paper. The figure shows the starter extract's descriptive freshness buckets, including their sample counts.


In [3]:
action_map = pd.DataFrame([
    {"priority":1,"archetype":"Top 50 by grouped model score","reason_code":"TOP_50_MODEL_REVIEW",
     "action":"Human editor inspects the page and records keep / revise / defer with evidence.",
     "evidence":"One fixed client-held-out split: Precision@50 0.720 versus proxy base rate 0.577.",
     "limit":"Trailing proxy; not future performance, not calibrated, not causal."},
    {"priority":2,"archetype":"91-180 days since update","reason_code":"FRESHNESS_AUDIT",
     "action":"Check facts, sources, intent, and technical status before deciding whether to edit.",
     "evidence":"Full starter snapshot proxy rate 0.611 (n=9,171) versus 0.512 for 0-90 days (n=20,655).",
     "limit":"Mixed, non-monotonic association; not a causal refresh effect."},
    {"priority":3,"archetype":"181-365 days since update","reason_code":"AGE_CAP_CHECK",
     "action":"Review freshness only when the model or editor flags another concrete issue.",
     "evidence":"Proxy rate 0.467 (n=169); age score contribution is capped at 180 days.",
     "limit":"Small bucket; do not infer that older pages are safer."},
    {"priority":4,"archetype":"366+ days since update","reason_code":"SMALL_N_CAUTION",
     "action":"Verify the update date and inspect manually; do not prioritize from this bucket rate.",
     "evidence":"Only 5 rows in the full starter snapshot.",
     "limit":"Too few observations for a stable rate."},
    {"priority":5,"archetype":"Fewer than 100 impressions in 90 days","reason_code":"OUTSIDE_EVALUATED_POPULATION",
     "action":"Use a separate discovery or measurement workflow; this model provides no score.",
     "evidence":"Excluded by the pre-existing ML-07 eligibility floor.",
     "limit":"Do not extrapolate the model to this population."},
    {"priority":6,"archetype":"Unknown or missing update age","reason_code":"MISSING_DATA_REVIEW",
     "action":"Resolve the source date before making an age-based decision.",
     "evidence":"No valid age signal.",
     "limit":"Missingness is not evidence of staleness."}
])
assert action_map["priority"].is_monotonic_increasing
assert action_map["reason_code"].is_unique

OUT_DIR = REPO_ROOT / "work" / "outputs"
FIG_DIR = REPO_ROOT / "work" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_PATH = OUT_DIR / "w07_ranked_content_action_queue.csv"
MAP_PATH = OUT_DIR / "w07_archetype_action_map.csv"
METRICS_PATH = OUT_DIR / "w07_action_playbook_metrics.json"
FIGURE_PATH = FIG_DIR / "w07_refresh_signal.svg"

ranked_queue.to_csv(QUEUE_PATH, index=False)
action_map.to_csv(MAP_PATH, index=False)

all_age = pd.to_numeric(df["days_since_last_update"], errors="coerce")
all_freshness = pd.cut(
    all_age, bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "366+"], right=True
)
age_table = (
    pd.DataFrame({"bucket": all_freshness.astype("string"), "decline_proxy": df["decline_proxy"]})
    .groupby("bucket", dropna=False, observed=False)
    .agg(n=("decline_proxy", "size"), proxy_rate=("decline_proxy", "mean"))
    .reindex(["0-90", "91-180", "181-365", "366+"])
    .reset_index()
)
age_table["proxy_rate"] = age_table["proxy_rate"].round(3)
print("Freshness signal from the full starter slice (descriptive proxy only):")
for row in age_table.itertuples(index=False):
    print(f"  {row.bucket:>7}: rate={row.proxy_rate:.3f}, n={row.n:,}")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_table = age_table.dropna(subset=["proxy_rate"]).copy()
bars = ax.bar(plot_table["bucket"], plot_table["proxy_rate"], color=["#00a99d", "#6272a4", "#9b5cff", "#ff2d78"])
ax.set_ylim(0, 0.75)
ax.set_ylabel("Observed trailing decline-proxy rate")
ax.set_xlabel("Days since last update")
ax.set_title("Freshness buckets: mixed, non-monotonic signal")
ax.grid(axis="y", alpha=0.2)
for bar, row in zip(bars, plot_table.itertuples(index=False)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
            f"{row.proxy_rate:.3f}\nn={row.n:,}", ha="center", va="bottom", fontsize=9)
fig.text(0.01, 0.01, "Starter extract only; trailing proxy, not a future outcome. 366+ bucket has n=5.", fontsize=8)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIGURE_PATH, format="svg", bbox_inches="tight")
plt.close(fig)

metrics_receipt = {
    "assignment":"ML-10",
    "lane":"Refresh / Content Opportunity Scoring",
    "dataset":"Repository starter extract; 30,000 anonymized content rows",
    "eligibility":"impressions_90d >= 100",
    "eligible_rows":int(len(eligible)),
    "train_rows":int(len(train_idx)),
    "test_rows":int(len(test_idx)),
    "train_clients":int(len(set(groups[train_idx]))),
    "test_clients":int(len(set(groups[test_idx]))),
    "client_overlap":False,
    "target":"trend_direction == down; trailing-window descriptive proxy, not a future outcome",
    "features":FEATURES,
    "ranking_score_note":"Logistic Regression output used only to order editor review; not calibrated.",
    "test_proxy_base_rate":round(test_base_rate,6),
    "model_metrics":{key:round(value,6) for key,value in model_metrics.items()},
    "week4_rule_same_test_metrics":{key:round(value,6) for key,value in rule_metrics.items()},
    "top50_review_tranche":{
        "precision_at_50":round(model_metrics["precision_at_50"],6),
        "base_rate":round(test_base_rate,6),
        "interpretation":"One fixed held-out-client split against the trailing proxy; directional only."
    },
    "full_snapshot_freshness_buckets":[
        {"bucket":str(row.bucket),"n":int(row.n),"proxy_rate":None if pd.isna(row.proxy_rate) else round(float(row.proxy_rate),6)}
        for row in age_table.itertuples(index=False)
    ],
    "monitoring_thresholds":{
        "status":"proposed, not evaluated in production",
        "pause_if_precision_at_50_below":round(test_base_rate,6),
        "investigate_if_below_reference_in_consecutive_independent_windows":2,
        "retrain_only_after":"a documented future outcome period and grouped/time-based evaluation"
    },
    "cost_value_note":"Review cost and realized refresh value were not measured; record editorial time, disposition, and later independent outcomes.",
    "exports":{
        "ranked_queue":"work/outputs/w07_ranked_content_action_queue.csv (regenerated; not committed)",
        "archetype_map":"work/outputs/w07_archetype_action_map.csv (regenerated; not committed)",
        "metrics_receipt":"work/outputs/w07_action_playbook_metrics.json",
        "figure":"work/figures/w07_refresh_signal.svg"
    },
    "software":{"pandas":pd.__version__,"numpy":np.__version__,"scikit_learn":sklearn.__version__}
}
METRICS_PATH.write_text(json.dumps(metrics_receipt,indent=2)+"\n",encoding="utf-8")
print(f"Saved ranked queue ({len(ranked_queue):,} rows): {QUEUE_PATH.relative_to(REPO_ROOT)}")
print(f"Saved archetype/action map ({len(action_map)} rows): {MAP_PATH.relative_to(REPO_ROOT)}")
print(f"Saved metrics receipt: {METRICS_PATH.relative_to(REPO_ROOT)}")
print(f"Saved reusable SVG figure: {FIGURE_PATH.relative_to(REPO_ROOT)}")


Freshness signal from the full starter slice (descriptive proxy only):
     0-90: rate=0.512, n=20,655
   91-180: rate=0.611, n=9,171
  181-365: rate=0.467, n=169
     366+: rate=0.600, n=5
Saved ranked queue (4,610 rows): work/outputs/w07_ranked_content_action_queue.csv
Saved archetype/action map (6 rows): work/outputs/w07_archetype_action_map.csv
Saved metrics receipt: work/outputs/w07_action_playbook_metrics.json
Saved reusable SVG figure: work/figures/w07_refresh_signal.svg


## 6. Self-check

This is decision support, not a production system. The queue is useful only for the observed starter extract and the evaluated eligible subset. Keep row-level CSVs out of Git; the notebook recreates them. The committed metrics JSON is the paper's small result receipt, and the SVG is the reusable freshness-signal figure.


In [4]:
assert len(ranked_queue) == 4_610
assert ranked_queue["content_id"].is_unique
assert ranked_queue["rank"].is_monotonic_increasing
assert ranked_queue.loc[ranked_queue["rank"] <= 50, "action_label"].eq("editor_review").all()
assert set(FEATURES).isdisjoint(FORBIDDEN)
assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))
assert len(action_map) == 6
assert QUEUE_PATH.exists() and QUEUE_PATH.stat().st_size > 0
assert MAP_PATH.exists() and MAP_PATH.stat().st_size > 0
assert METRICS_PATH.exists() and METRICS_PATH.stat().st_size > 0
assert FIGURE_PATH.exists() and FIGURE_PATH.stat().st_size > 0
assert np.isclose(model_metrics["precision_at_50"], 0.720, atol=0.001)
assert np.isclose(test_base_rate, 0.577, atol=0.001)

print("PASS: notebook rebuilt the fixed Week-5 grouped split; train/test clients are disjoint.")
print("PASS: leakage guard excludes IDs, target fields, and overlapping traffic windows from features.")
print("PASS: top-50 action is human review; no refresh, publish, or deletion is automated.")
print("PASS: generated queue and map exist; CSVs remain excluded from Git by repository rules.")
print("PASS: committed metrics receipt and reusable SVG figure exist.")
print("PASS: one split / trailing proxy / unmeasured impact limits are recorded.")
print("All six sections are complete.")


PASS: notebook rebuilt the fixed Week-5 grouped split; train/test clients are disjoint.
PASS: leakage guard excludes IDs, target fields, and overlapping traffic windows from features.
PASS: top-50 action is human review; no refresh, publish, or deletion is automated.
PASS: generated queue and map exist; CSVs remain excluded from Git by repository rules.
PASS: committed metrics receipt and reusable SVG figure exist.
PASS: one split / trailing proxy / unmeasured impact limits are recorded.
All six sections are complete.
